# 📊 Exploration des Donnees Financieres

Ce notebook explore les donnees collectees et nettoyees du projet IA Finance.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Chargement des donnees
df = pd.read_csv('../data/processed/all_tickers_clean.csv', index_col=0, parse_dates=True)
print(f'Shape: {df.shape}')
print(f'Tickers: {df["Ticker"].unique().tolist()}')
print(f'Periode: {df.index.min()} -> {df.index.max()}')
df.head()

## 1. Apercu general des donnees

In [ ]:
# Statistiques descriptives
df.describe().round(2)

In [ ]:
# Valeurs manquantes
missing = df.isnull().sum()
print('Valeurs manquantes par colonne:')
print(missing[missing > 0])

## 2. Evolution des prix

In [ ]:
# Evolution du prix de cloture par ticker
fig = px.line(df.reset_index(), x='Date', y='Close', color='Ticker',
             title='Evolution des prix de cloture')
fig.update_layout(height=500)
fig.show()

In [ ]:
# Performance normalisee (base 100)
fig = go.Figure()
for ticker in df['Ticker'].unique():
    data = df[df['Ticker'] == ticker]['Close']
    normalized = (data / data.iloc[0]) * 100
    fig.add_trace(go.Scatter(x=data.index, y=normalized, name=ticker, mode='lines'))

fig.update_layout(title='Performance normalisee (base 100)', 
                  yaxis_title='Performance', height=500)
fig.show()

## 3. Distribution des rendements

In [ ]:
# Distribution des rendements journaliers
fig = px.histogram(df, x='Return_1d', color='Ticker', 
                   title='Distribution des rendements journaliers',
                   nbins=100, opacity=0.7)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Box plot des rendements par ticker
fig = px.box(df, x='Ticker', y='Return_1d', 
             title='Rendements journaliers par ticker')
fig.show()

## 4. Volatilite

In [ ]:
# Evolution de la volatilite
fig = px.line(df.reset_index(), x='Date', y='Volatility_20d', color='Ticker',
             title='Volatilite sur 20 jours')
fig.update_layout(height=400)
fig.show()

In [ ]:
# Volatilite moyenne par ticker
vol_mean = df.groupby('Ticker')['Volatility_20d'].mean().sort_values(ascending=False)
fig = px.bar(x=vol_mean.index, y=vol_mean.values,
             title='Volatilite moyenne par ticker',
             labels={'x': 'Ticker', 'y': 'Volatilite moyenne'})
fig.show()

## 5. Correlations

In [ ]:
# Matrice de correlation des rendements
returns_pivot = df.pivot_table(values='Return_1d', index=df.index, columns='Ticker')
correlation = returns_pivot.corr()

fig = px.imshow(correlation, text_auto='.2f',
               title='Correlation des rendements entre tickers',
               color_continuous_scale='RdBu_r')
fig.show()

## 6. Indicateurs techniques

In [ ]:
# RSI pour un ticker
ticker = 'AAPL'
data = df[df['Ticker'] == ticker].tail(250)

fig = go.Figure()
fig.add_trace(go.Scatter(x=data.index, y=data['RSI_14'], name='RSI 14'))
fig.add_hline(y=70, line_dash='dash', line_color='red', annotation_text='Suracheté')
fig.add_hline(y=30, line_dash='dash', line_color='green', annotation_text='Survendu')
fig.update_layout(title=f'RSI 14 jours — {ticker}', height=350)
fig.show()

In [ ]:
# Moyennes mobiles + prix
fig = go.Figure()
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], name='Prix', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=data.index, y=data['MA_7'], name='MA 7', line=dict(dash='dot')))
fig.add_trace(go.Scatter(x=data.index, y=data['MA_20'], name='MA 20', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=data.index, y=data['MA_50'], name='MA 50', line=dict(dash='dashdot')))
fig.update_layout(title=f'Prix et Moyennes Mobiles — {ticker}', height=450)
fig.show()

## 7. Volume d'echange

In [ ]:
# Volume moyen par ticker
vol_avg = df.groupby('Ticker')['Volume'].mean().sort_values(ascending=False)
fig = px.bar(x=vol_avg.index, y=vol_avg.values,
             title='Volume moyen journalier par ticker',
             labels={'x': 'Ticker', 'y': 'Volume moyen'})
fig.show()